# Course project

This notebook includes the code for the course project in the DTU course [Integrated Energy Grids](https://kurser.dtu.dk/course/2024-2025/46770?menulanguage=en)  and is modelling and analyzing the german energy system. 
 
The report is provided along this notebook, further explaining the results. 

### Imports

In [2]:
import pandas as pd
import pypsa
import matplotlib.pyplot as plt
import numpy as np

In [3]:
import os
import sys

# Set path to the directory containing base.py
module_path = os.path.abspath(".")

if module_path not in sys.path:
    sys.path.append(module_path)

import base  # now you can use base.build_network()

from visualization import *

In [4]:
# how to use base module:

# 1. Initilize Base model
# network = base.build_network(solve=False)  # just builds it

# 2. Edit/add components based on assignment details
# do some edits...

# 3. Solve model
# network.optimize(solver_name="highs")  # solve manually later

## E. Decarbonization
Select one target for decarbonization (i.e., one CO2 allowance limit). What is the CO2 price required to achieve that decarbonization level? Search for information on the existing CO2 tax in your country (if any) and discuss your results.

In [5]:
network = base.build_network(solve=False)

/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:26: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  snapshots = pd.date_range(f"{year}-01-01", f"{year}-12-31 23:00", freq="H")
/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_daily["Inflow [MW]"] = df_daily["Inflow [GWh]"] * 1000 / 24  # MW average per hour
/Users/jonaswiendl/Documents/46770_IEG/46770_IEG/base.py:96: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  inflow_ror_hourly = df_daily["Inflow [MW]"].resample("H").interpolate("linear")


Add CO2 constraint of german emissions in 2015

In [11]:
co2_emissions_1990 = 366e6
co2_allowance_2040 = 0.12*co2_emissions_1990
co2_allowance_2040/10e6

4.392

In [7]:
network.add("GlobalConstraint", "CO2Limit",
          carrier_attribute="co2_emissions",
          sense="<=",
          constant=co2_allowance_2040)

network.optimize(solver_name="highs")

Index(['DEU_elec'], dtype='object', name='Bus')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 2/2 [00:00<00:00, 28.53it/s]
INFO:linopy.io: Writing time: 0.58s


Running HiGHS 1.9.0 (git hash: n/a): Copyright (c) 2024 HiGHS under MIT licence terms
Coefficient ranges:
  Matrix [1e-03, 1e+00]
  Cost   [1e-02, 4e+05]
  Bound  [0e+00, 0e+00]
  RHS    [5e+03, 4e+07]
Presolving model
109633 rows, 65840 cols, 293856 nonzeros  0s
100873 rows, 57080 cols, 324648 nonzeros  0s
Presolve : Reductions: rows 100873(-48058); columns 57080(-13008); elements 324648(-12754)
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -1.4471818664e+09 Ph1: 100873(1.70739e+08); Du: 57072(1.54681e+06) 0s
       9909    -3.5300529014e+08 Ph1: 89083(9.4965e+07); Du: 43781(3.84986e+06) 5s
      14107    -3.5166677066e+08 Ph1: 84749(7.10234e+07); Du: 46350(3.77116e+06) 11s
      17882    -3.4583400174e+08 Ph1: 77986(5.40708e+07); Du: 45734(3.77694e+06) 16s
      21779    -3.2267646065e+08 Ph1: 71026(3.53357e+07); Du: 44202(2.72003e+06) 21s
      29279    -2.2894227866e+07 Ph1: 59655(4.80287e+06

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 70088 primals, 148931 duals
Objective: 3.76e+10
Solver model: available
Solver message: optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper were not assigned to the network.


('ok', 'optimal')

In [9]:
# check some numbers

print(f"System cost: {round(network.objective / 1e9, 2)} billion euros")
print("")
print("Optimal Generator Capactities in GW:")
print(network.generators.p_nom_opt.div(1e3))  # MW -> GW
print("")
print("Optimal Energy Generation in GWh/a")
print(network.generators_t.p.sum().div(1e6))

# Total emissions in the system (sum over all generators and time)
actual_emissions = (
    network.generators_t.p
    .multiply(network.generators.carrier.map(network.carriers["co2_emissions"]))
    .sum()
    .sum()
)

print("")
# The imposed CO₂ limit (from GlobalConstraint)
co2_limit = network.global_constraints.at["CO2Limit", "constant"]

print(f"Actual emissions: {actual_emissions:.2e} t")
print(f"CO₂ constraint  : {co2_limit:.2e} t")

dual = network.global_constraints.at["CO2Limit", "mu"]
print(f"Shadow price of CO₂: {dual:.2f} €/tCO2")

System cost: 37.6 billion euros

Optimal Generator Capactities in GW:
Generator
coal           -0.000000
lignite        -0.000000
biomass CHP     5.000000
OCGT           66.143381
ror             5.647002
onwind         20.827833
offwind        47.685581
solar          74.891179
Name: p_nom_opt, dtype: float64

Optimal Energy Generation in GWh/a
Generator
coal             0.000000
lignite          0.000000
biomass CHP     34.329163
OCGT           223.636364
ror             15.477029
onwind          21.343849
offwind        134.406264
solar           76.071895
dtype: float64

Actual emissions: 4.43e+07 t
CO₂ constraint  : 4.43e+07 t
Shadow price of CO₂: -365.22 €/tCO2
